# The catalog as a graph

Four cells, one story: the catalog is a list, then a card, then a graph, then a query.

Needs a running kernel (`ikigai serve /tmp/ikbook-kernel.sock`) and two packages:
`pip install rdflib /path/to/ikigai-python`. `examples/python/catalog.py` is these
same cells as a script, which is what the book's tests run.

In [ ]:
import ikigai, rdflib

k = ikigai.connect("/tmp/ikbook-kernel.sock")

# 1. The catalog as a list: every name the kernel binds, and the endpoint behind it.
entries = k.entries()
print(len(entries), "bound names; the first three:")
for entry in entries[:3]:
    print(" ", entry.pattern, "→", entry.endpoint)

In [ ]:
# 2. One endpoint's card, as data — the JSON Meta face, parsed.
card = k.describe("urn:iki:fn:toUpper")
print(card["title"], "—", card["summary"])
for arg in card["inputs"]:
    print("  input", arg["name"], "required" if arg.get("required", True) else "optional")

In [ ]:
# 3. The whole catalog as a graph. urn:kernel:catalog answers as Turtle; rdflib parses it.
turtle = k.source("urn:kernel:catalog").text
g = rdflib.Graph()
g.parse(data=turtle, format="turtle")
print(len(g), "triples in the catalog")

In [ ]:
# 4. One SPARQL query over it: which endpoints take a string?
query = """
    PREFIX ik: <https://ikigai-rs.dev/ns#>
    SELECT DISTINCT ?id WHERE {
      ?endpoint a ik:Endpoint ; ik:id ?id ; (ik:input | ik:action/ik:input) ?input .
      ?input ik:class <http://www.w3.org/2001/XMLSchema#string> .
    } ORDER BY ?id
"""
for row in g.query(query):
    print(" ", row.id)

k.close()